# Kandinsky 6 SR — Diffusers example

Runs Kandinsky 6.0 video super-resolution through 🧨 Diffusers (`Kandinsky6SRPipeline`, available from `diffusers` v0.42.0) on one video file and saves the result. The Diffusers bundle holds the π-Flow distilled SR DiT, the KVAE and the x2 / x4 latent upscalers; it is downloaded into the HF cache on first use.

This notebook needs only Diffusers, not the `kandinsky-6-sr` package:

```bash
pip install "diffusers>=0.42.0" transformers accelerate imageio imageio-ffmpeg
```

The Diffusers pipeline is video-only and does not resample the frame rate: feed a 24 fps clip and mux the source audio back in yourself if you need it. For fps resampling, audio passthrough and the faster KVAE / NABLA backends, see `sr_inference_example.ipynb`.

### 1. Config

The `TORCHINDUCTOR_*` variables are read once, when torch's inductor config is first imported (by the first `diffusers` import below), so this cell runs before any of them — restart the kernel after changing them. They keep `max_autotune` (enabled in step 3) from trying kernel configs that exceed the GPU's shared memory (the `No valid triton configs. OutOfMemoryError: out of resource` errors) and keep its timing tables out of the output.

In [ ]:
import os
from pathlib import Path

import torch
from IPython.display import Video, display

os.environ["TORCHINDUCTOR_MAX_AUTOTUNE_PRUNE_CHOICES_BASED_ON_SHARED_MEM"] = "1"  # skip configs the GPU cannot launch
os.environ["TORCHINDUCTOR_AUTOTUNE_NUM_CHOICES_DISPLAYED"] = "0"  # no "AUTOTUNE addmm(...)" timing tables
os.environ["TORCHINDUCTOR_MAX_AUTOTUNE_REPORT_CHOICES_STATS"] = "0"  # no "Autotune Choices Stats" lines

MODEL_ID = "kandinskylab/Kandinsky-6.0-VSR-2.0-distilled2steps-Diffusers"
INPUT_VIDEO = Path("inputs/lq.mp4")  # a 24 fps clip of up to 15 s
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RESOLUTION_SCALE = 2  # 2, 4, or 2.25 (x1.125 bilinear pre-upscale + x2 tiling)
NUM_INFERENCE_STEPS = 2  # the distilled bundle runs 2 steps; the pipeline default (4) is for the flow-matching one
MAX_NUM_FRAMES = 361  # 15 s at 24 fps
FPS = 24
SEED = 1137

### 2. Load the input video

`load_video` returns a list of PIL frames. The pipeline needs `1 + 4k` frames (the KVAE compresses time by 4), so the clip is cut to the 15 s budget and floor-aligned to that grid.

In [ ]:
from diffusers.utils import load_video

frames = load_video(str(INPUT_VIDEO))[:MAX_NUM_FRAMES]
frames = frames[: 1 + 4 * ((len(frames) - 1) // 4)]
print(f"input: {len(frames)} frames, {frames[0].width}x{frames[0].height}")
display(Video(str(INPUT_VIDEO), embed=False, width=640))

### 3. Load the pipeline

The SR transformer runs NABLA sparse attention on the `flex` backend; compiling the repeated blocks keeps flex off its memory-hungry eager fallback, and `max_autotune` lets inductor pick flex-attention tiles that fit the SR block mask. Model CPU offload keeps only the active component on the GPU.

In [ ]:
from diffusers import Kandinsky6SRPipeline

torch._inductor.config.max_autotune = True

pipe = Kandinsky6SRPipeline.from_pretrained(MODEL_ID, dtype=torch.bfloat16)
pipe.enable_model_cpu_offload()
pipe.transformer.set_attention_backend("flex")
pipe.transformer.compile_repeated_blocks(fullgraph=True)

### 4. Inference

Tiled SR: the video is split into overlapping tiles, each refined at a trained tile size, and blended back with Hann windows. The first tile also pays for compilation.

In [ ]:
from diffusers.utils import export_to_video

with torch.inference_mode():
    output = pipe(
        video=frames,
        resolution_scale=RESOLUTION_SCALE,
        num_inference_steps=NUM_INFERENCE_STEPS,
        generator=torch.Generator("cuda").manual_seed(SEED),
    )

sr_frames = output.frames[0]
sr_path = OUT_DIR / f"{INPUT_VIDEO.stem}_sr_{str(RESOLUTION_SCALE).replace('.', 'p')}x_diffusers.mp4"
export_to_video(sr_frames, str(sr_path), fps=FPS)
print(f"SR output: {len(sr_frames)} frames, {sr_frames[0].width}x{sr_frames[0].height} -> {sr_path}")

In [ ]:
display(Video(str(sr_path), embed=False, width=640))